# Lung GMM Interactive Batch Processing

DICOM CT を1症例ずつ読み込み、**手動で ROI（肺領域）を調整** してから GMM 解析を実行するノートブック。

## ワークフロー

1. **Cell 1-3**: 初期設定（最初に1回だけ実行）
2. **Cell 4**: 次の患者を読み込み → Slicer に ROI ボックスが表示される
3. **Slicer 画面で ROI を調整** → 肺領域のみを囲むようにドラッグ
4. **Cell 5**: ROI でクロップ → GMM 解析 → 結果保存
5. Cell 4-5 を全症例分繰り返す
6. **Cell 6**: 全症例の集約 CSV を出力

## 前提条件

- 3D Slicer 5.x + SlicerJupyter 拡張機能
- SlicerDensityLungSegmentation（なければ fallback モードで動作）

---
## 1. 設定

In [4]:
# ============================================================
# ユーザー設定
# ============================================================
INPUT_DIR = "/path/to/Hosogaya_BAL/ILD Data"
OUTPUT_DIR = "/path/to/CT_results/gmm_interactive"

FALLBACK_MODE = False  # 拡張機能が動かない場合のみ True

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Input:  /path/to/Hosogaya_BAL/ILD Data
Output: /path/to/CT_results/gmm_interactive


---
## 2. ヘルパー関数の定義

In [5]:
import slicer
import os
import json
import csv
import glob
import time
import numpy as np
from datetime import datetime

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# DICOM データベース初期化
# ============================================================
def ensure_dicom_database():
    """Slicer の DICOM データベースを初期化"""
    if not slicer.dicomDatabase or not slicer.dicomDatabase.isOpen:
        db_dir = os.path.join(slicer.app.temporaryPath, "SlicerDICOMDatabase")
        os.makedirs(db_dir, exist_ok=True)
        db_path = os.path.join(db_dir, "ctkDICOM.sql")
        dicomBrowser = slicer.modules.dicom.widgetRepresentation().self()
        dicomBrowser.browserWidget.dicomBrowser.setDatabaseDirectory(db_dir)
        slicer.dicomDatabase.openDatabase(db_path)
    print("DICOM database: ready")


# ============================================================
# DICOM ディレクトリの検索
# ============================================================
def find_dicom_patients(input_dir):
    """ILD Data から DICOM 患者ディレクトリを検索"""
    patients = []
    for item in sorted(os.listdir(input_dir)):
        item_path = os.path.join(input_dir, item)
        if not os.path.isdir(item_path):
            continue
        dcmdt_path = os.path.join(item_path, "DCMDT")
        if os.path.isdir(dcmdt_path):
            n_files = len([f for f in os.listdir(dcmdt_path)
                           if os.path.isfile(os.path.join(dcmdt_path, f))
                           and not f.startswith('.')])
            if n_files > 0:
                patients.append({
                    "name": item,
                    "path": dcmdt_path,
                    "n_slices": n_files
                })
    return patients


# ============================================================
# DICOM からボリューム読み込み
# ============================================================
def load_volume_from_dicom(dicom_dir):
    """DICOM ディレクトリからボリュームノードを読み込む"""
    from DICOMLib import DICOMUtils

    DICOMUtils.importDicom(dicom_dir)
    slicer.app.processEvents()

    db = slicer.dicomDatabase
    patient_uids = db.patients()
    if not patient_uids:
        return None

    # 対象ディレクトリ内のシリーズを収集
    dicom_dir_abs = os.path.abspath(dicom_dir)
    candidate_series = []

    for patient_uid in patient_uids:
        for study_uid in db.studiesForPatient(patient_uid):
            for series_uid in db.seriesForStudy(study_uid):
                files = db.filesForSeries(series_uid)
                if not files:
                    continue
                if os.path.abspath(files[0]).startswith(dicom_dir_abs):
                    candidate_series.append({
                        "series_uid": series_uid,
                        "n_files": len(files)
                    })

    if not candidate_series:
        latest_patient = patient_uids[-1]
        for study_uid in db.studiesForPatient(latest_patient):
            for series_uid in db.seriesForStudy(study_uid):
                files = db.filesForSeries(series_uid)
                if files:
                    candidate_series.append({
                        "series_uid": series_uid,
                        "n_files": len(files)
                    })

    if not candidate_series:
        return None

    # 最大スライス数のシリーズを選択
    candidate_series.sort(key=lambda x: x["n_files"], reverse=True)
    best = candidate_series[0]
    print(f"  Series: {best['series_uid']} ({best['n_files']} slices)")

    loaded_ids = DICOMUtils.loadSeriesByUID([best["series_uid"]])
    slicer.app.processEvents()

    if not loaded_ids:
        return None

    for node_id in loaded_ids:
        node = slicer.mrmlScene.GetNodeByID(node_id)
        if node and node.IsA("vtkMRMLScalarVolumeNode"):
            return node

    return slicer.mrmlScene.GetFirstNodeByClass("vtkMRMLScalarVolumeNode")


# ============================================================
# ROI の作成と表示
# ============================================================
def create_lung_roi(volume_node):
    """
    ボリュームの範囲に基づいて初期 ROI を作成。
    Z軸は中央80%に設定（肺尖〜肺底部の概算）。
    ユーザーが Slicer 画面上でドラッグして調整する。
    """
    bounds = [0] * 6
    volume_node.GetBounds(bounds)
    # bounds: [xmin, xmax, ymin, ymax, zmin, zmax]

    center_x = (bounds[0] + bounds[1]) / 2
    center_y = (bounds[2] + bounds[3]) / 2
    center_z = (bounds[4] + bounds[5]) / 2

    size_x = bounds[1] - bounds[0]
    size_y = bounds[3] - bounds[2]
    size_z = bounds[5] - bounds[4]

    # ROI: X,Y は全幅、Z は中央80%（上下10%をカット）
    roi_node = slicer.mrmlScene.AddNewNodeByClass(
        "vtkMRMLMarkupsROINode", "LungROI"
    )
    roi_node.SetCenter(center_x, center_y, center_z)
    roi_node.SetSize(size_x, size_y, size_z * 0.8)

    # 表示設定
    roi_node.GetDisplayNode().SetSelectedColor(0.0, 1.0, 0.0)
    roi_node.GetDisplayNode().SetHandleVisibility(True)
    roi_node.GetDisplayNode().SetFillVisibility(True)
    roi_node.GetDisplayNode().SetFillOpacity(0.1)

    print(f"  ROI created: center=({center_x:.0f}, {center_y:.0f}, {center_z:.0f})")
    print(f"  ROI size: {size_x:.0f} x {size_y:.0f} x {size_z*0.8:.0f} mm")

    return roi_node


# ============================================================
# CropVolume でクロップ
# ============================================================
def crop_volume_with_roi(volume_node, roi_node):
    """ROI でボリュームをクロップして新しいボリュームを返す"""
    crop_params = slicer.mrmlScene.AddNewNodeByClass(
        "vtkMRMLCropVolumeParametersNode"
    )
    crop_params.SetInputVolumeNodeID(volume_node.GetID())
    crop_params.SetROINodeID(roi_node.GetID())
    crop_params.SetVoxelBased(True)
    crop_params.SetInterpolationMode(
        crop_params.InterpolationNearestNeighbor
    )

    slicer.modules.cropvolume.logic().Apply(crop_params)
    slicer.app.processEvents()

    cropped_node = slicer.mrmlScene.GetNodeByID(
        crop_params.GetOutputVolumeNodeID()
    )
    slicer.mrmlScene.RemoveNode(crop_params)

    if cropped_node:
        arr = slicer.util.arrayFromVolume(cropped_node)
        print(f"  Cropped volume: {arr.shape} voxels")

    return cropped_node


# ============================================================
# GMM 解析（拡張機能 or フォールバック）
# ============================================================
def run_gmm_analysis(volume_node, use_extension=True):
    """GMM による肺組織分類を実行"""
    if use_extension:
        slicer.util.selectModule("LungCTGMMSegmentation")
        slicer.app.processEvents()

        module = slicer.modules.lungctgmmsegmentation
        widget = module.widgetRepresentation().self()

        output_seg = slicer.mrmlScene.AddNewNodeByClass(
            "vtkMRMLSegmentationNode", "GMM_Result"
        )
        averaged_seg = slicer.mrmlScene.AddNewNodeByClass(
            "vtkMRMLSegmentationNode", "GMM_Averaged"
        )

        widget.ui.inputVolumeSelector.setCurrentNode(volume_node)
        widget.ui.outputSegmentationSelector.setCurrentNode(output_seg)
        widget.ui.outputAveragedSegmentationSelector.setCurrentNode(averaged_seg)

        widget.onApplyButton()
        slicer.app.processEvents()
        time.sleep(2)

        return output_seg
    else:
        return run_fallback_gmm(volume_node)


def run_fallback_gmm(volume_node):
    """論文パラメータで GMM を再学習して適用"""
    from sklearn.mixture import GaussianMixture
    from scipy import ndimage
    from scipy.ndimage import median_filter

    volume_array = slicer.util.arrayFromVolume(volume_node)
    print(f"  Volume shape: {volume_array.shape}")

    # 肺領域セグメンテーション
    print("  1/6 Threshold (-155 HU)...")
    lung_mask = volume_array < -155

    print("  2/6 Island removal...")
    labeled_array, num_features = ndimage.label(lung_mask)
    if num_features > 0:
        sizes = ndimage.sum(lung_mask, labeled_array, range(1, num_features + 1))
        if len(sizes) > 2:
            threshold = sorted(sizes, reverse=True)[1] * 0.1
            for i, size in enumerate(sizes):
                if size < threshold:
                    lung_mask[labeled_array == (i + 1)] = False

    print("  3/6 Morphological closing...")
    struct = ndimage.generate_binary_structure(3, 2)
    lung_mask = ndimage.binary_closing(lung_mask, structure=struct, iterations=3)
    lung_mask = ndimage.binary_fill_holes(lung_mask)
    print(f"      Lung voxels: {np.sum(lung_mask):,}")

    print("  4/6 GMM fitting (5 components)...")
    lung_intensities = volume_array[lung_mask].astype(np.float64).reshape(-1, 1)
    gmm = GaussianMixture(
        n_components=5, covariance_type='full', n_init=6,
        init_params='kmeans', random_state=42, max_iter=200, tol=1e-3
    )
    gmm.fit(lung_intensities)
    print(f"      Converged: {gmm.converged_}")

    print("  5/6 MAP estimation...")
    raw_labels = gmm.predict(lung_intensities)
    means = gmm.means_.flatten()
    sorted_indices = np.argsort(means)
    label_mapping = {old: new for new, old in enumerate(sorted_indices)}
    mapped_labels = np.array([label_mapping[l] for l in raw_labels])

    label_map = np.zeros(volume_array.shape, dtype=np.int8)
    label_map[lung_mask] = mapped_labels + 1

    tissue_names = ["Air", "Healthy_Lung", "GGO", "Consolidation", "Dense_Tissue"]
    sorted_means = means[sorted_indices]
    for i, (name, mean) in enumerate(zip(tissue_names, sorted_means)):
        print(f"      {i}: {name} = {mean:.1f} HU")

    print("  6/6 Median filter...")
    mask_nz = label_map > 0
    filtered = median_filter(label_map, size=4)
    label_map[mask_nz] = filtered[mask_nz]

    # セグメンテーションノード作成
    colors = {1:[0,0,0], 2:[0,0.5,1], 3:[1,1,0], 4:[1,0.5,0], 5:[1,0,0]}

    labelmapNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode")
    labelmapNode.CopyOrientation(volume_node)
    slicer.util.updateVolumeFromArray(labelmapNode, label_map)
    labelmapNode.SetOrigin(volume_node.GetOrigin())
    labelmapNode.SetSpacing(volume_node.GetSpacing())
    ijkToRas = slicer.vtkMatrix4x4()
    volume_node.GetIJKToRASMatrix(ijkToRas)
    labelmapNode.SetIJKToRASMatrix(ijkToRas)

    output_seg = slicer.mrmlScene.AddNewNodeByClass(
        "vtkMRMLSegmentationNode", "GMM_Result"
    )
    slicer.modules.segmentations.logic().ImportLabelmapToSegmentationNode(
        labelmapNode, output_seg
    )
    segmentation = output_seg.GetSegmentation()
    for i in range(segmentation.GetNumberOfSegments()):
        seg_id = segmentation.GetNthSegmentID(i)
        segment = segmentation.GetSegment(seg_id)
        lbl = i + 1
        if lbl <= len(tissue_names):
            segment.SetName(tissue_names[lbl - 1])
        if lbl in colors:
            segment.SetColor(*colors[lbl])
    slicer.mrmlScene.RemoveNode(labelmapNode)

    output_seg.CreateDefaultDisplayNodes()
    output_seg.GetDisplayNode().SetVisibility2D(True)

    return output_seg


# ============================================================
# Segment Statistics
# ============================================================
def compute_and_save(volume_node, seg_node, patient_name, output_dir):
    """統計計算、結果表示、ファイル保存を行い result_row を返す"""
    import SegmentStatistics

    segStatLogic = SegmentStatistics.SegmentStatisticsLogic()
    segStatLogic.getParameterNode().SetParameter("Segmentation", seg_node.GetID())
    segStatLogic.getParameterNode().SetParameter("ScalarVolume", volume_node.GetID())
    segStatLogic.computeStatistics()
    stats = segStatLogic.getStatistics()

    result_row = {"patient": patient_name}
    total_vol = 0

    for segId in stats["SegmentIDs"]:
        seg_name = stats[segId, "Segment"]
        vol = stats[segId, "LabelmapSegmentStatisticsPlugin.volume_cm3"]
        result_row[f"{seg_name}_cm3"] = round(vol, 2)
        total_vol += vol

    result_row["total_cm3"] = round(total_vol, 2)

    for segId in stats["SegmentIDs"]:
        seg_name = stats[segId, "Segment"]
        vol = stats[segId, "LabelmapSegmentStatisticsPlugin.volume_cm3"]
        pct = round(vol / total_vol * 100, 2) if total_vol > 0 else 0
        result_row[f"{seg_name}_pct"] = pct

    # 肺病変度
    ggo = consolidation = dense = healthy = 0
    for key, val in result_row.items():
        kl = key.lower()
        if key.endswith("_pct"):
            if "glass" in kl or "ggo" in kl:
                ggo = val
            elif "consolidat" in kl:
                consolidation = val
            elif "dense" in kl:
                dense = val
            elif "healthy" in kl:
                healthy = val

    result_row["lung_involvement_pct"] = round(ggo + consolidation + dense, 2)

    # 表示
    print(f"\n  --- Results: {patient_name} ---")
    for segId in stats["SegmentIDs"]:
        seg_name = stats[segId, "Segment"]
        vol = result_row.get(f"{seg_name}_cm3", 0)
        pct = result_row.get(f"{seg_name}_pct", 0)
        bar = '#' * int(pct / 2)
        print(f"  {seg_name:20s}: {vol:8.1f} cm3 ({pct:5.1f}%) {bar}")
    print(f"  {'Total':20s}: {total_vol:8.1f} cm3")
    print(f"  Lung involvement (I) = {result_row['lung_involvement_pct']:.1f}%")
    print(f"  Healthy lung         = {healthy:.1f}%")

    # 保存
    pt_dir = os.path.join(output_dir, patient_name)
    os.makedirs(pt_dir, exist_ok=True)

    slicer.util.saveNode(seg_node, os.path.join(pt_dir, "gmm_segmentation.seg.nrrd"))
    segStatLogic.exportToCSVFile(os.path.join(pt_dir, "segment_statistics.csv"))

    json_data = {
        "method": "Zaffino_et_al_2021_GMM",
        "patient": patient_name,
        "analysis_date": datetime.now().isoformat(),
        "slicer_version": slicer.app.applicationVersion,
        "roi_cropped": True,
        "lung_involvement_pct": result_row["lung_involvement_pct"],
        "results": {k: v for k, v in result_row.items() if k != "patient"}
    }
    with open(os.path.join(pt_dir, "gmm_results.json"), 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)

    print(f"  Saved to: {pt_dir}")
    return result_row


print("Helper functions loaded.")

ModuleNotFoundError: No module named 'slicer'

---
## 3. 患者リストの読み込み

In [3]:
# DICOM データベース初期化
ensure_dicom_database()

# 患者リスト取得
patients = find_dicom_patients(INPUT_DIR)
print(f"Found {len(patients)} patients:\n")
for i, p in enumerate(patients):
    print(f"  {i+1:3d}. {p['name']}  ({p['n_slices']} slices)")

# 拡張機能チェック
use_ext = not FALLBACK_MODE
if use_ext:
    try:
        _ = slicer.modules.lungctgmmsegmentation
        print(f"\nSlicerDensityLungSegmentation: detected")
    except AttributeError:
        print(f"\nExtension not found -> fallback mode")
        use_ext = False

# 進捗管理
current_index = 0
all_results = []
skipped = []

# 現在の状態を保持する変数
current_volume = None
current_roi = None
current_patient = None

print(f"\n--- Cell 4 を実行して最初の患者を読み込んでください ---")

NameError: name 'ensure_dicom_database' is not defined

---
## 4. 次の患者を読み込み + ROI 作成

**このセルを実行すると:**
1. 次の患者の DICOM を読み込みます
2. Slicer 画面に CT と **緑色の ROI ボックス** が表示されます

**表示されたら Slicer 画面で:**
- ROI ボックスの **ハンドル（角や辺の丸い点）をドラッグ** して肺領域のみを囲む
- 特に **Z軸（上下）** を調整して、頸部や腹部を除外する
- 気管が含まれる場合は上端を下げる

**調整が終わったら Cell 5 を実行**

In [ ]:
# === 次の患者を読み込む ===

if current_index >= len(patients):
    print("全患者の処理が完了しています。")
    print("Cell 6 を実行して集約 CSV を出力してください。")
else:
    current_patient = patients[current_index]
    print(f"{'='*60}")
    print(f"[{current_index + 1}/{len(patients)}] {current_patient['name']}")
    print(f"{'='*60}")

    # シーンをクリア
    slicer.mrmlScene.Clear(0)
    slicer.app.processEvents()
    time.sleep(1)

    # DICOM 読み込み
    print("Loading DICOM...")
    current_volume = load_volume_from_dicom(current_patient["path"])

    if current_volume:
        # スライスビューをフィット
        slicer.util.setSliceViewerLayers(background=current_volume)
        slicer.util.resetSliceViews()
        slicer.app.processEvents()

        # ROI 作成
        current_roi = create_lung_roi(current_volume)
        slicer.app.processEvents()

        print(f"\n>> Slicer 画面で ROI を肺領域に合わせて調整してください")
        print(f">> 調整後、Cell 5 を実行してください")
        print(f">> スキップする場合は Cell 5 の代わりに Cell 4 を再実行してください")
    else:
        print(f"  ERROR: ボリュームの読み込みに失敗")
        skipped.append(current_patient["name"])
        current_index += 1
        print(f"  Cell 4 を再実行して次の患者に進んでください")

---
## 5. ROI でクロップ → GMM 解析 → 保存

**ROI の調整が完了してからこのセルを実行してください。**

処理内容:
1. ROI でボリュームをクロップ
2. クロップ後のボリュームに GMM 解析を実行
3. 結果を表示・保存
4. 自動的に次の患者に進む（Cell 4 に戻って実行）

In [ ]:
# === ROI でクロップ → GMM 解析 → 保存 ===

if current_volume is None or current_roi is None:
    print("ERROR: 先に Cell 4 を実行して患者を読み込んでください")
elif current_patient is None:
    print("ERROR: 患者データがありません")
else:
    patient_name = current_patient["name"]
    print(f"Processing: {patient_name}")
    t0 = time.time()

    try:
        # 1. ROI でクロップ
        print("\n  [1/3] Cropping with ROI...")
        cropped_volume = crop_volume_with_roi(current_volume, current_roi)

        if cropped_volume is None:
            raise RuntimeError("Crop failed")

        # 2. GMM 解析
        print("\n  [2/3] Running GMM analysis...")
        seg_node = run_gmm_analysis(cropped_volume, use_extension=use_ext)

        if seg_node is None:
            raise RuntimeError("GMM analysis failed")

        # 3. 統計計算 + 保存
        print("\n  [3/3] Computing statistics...")
        result_row = compute_and_save(
            cropped_volume, seg_node, patient_name, OUTPUT_DIR
        )
        all_results.append(result_row)

        elapsed = time.time() - t0
        print(f"\n  Time: {elapsed:.1f}s")

    except Exception as e:
        print(f"\n  ERROR: {e}")
        import traceback
        traceback.print_exc()
        skipped.append(patient_name)

    # 次の患者へ進む
    current_index += 1
    current_volume = None
    current_roi = None
    current_patient = None

    print(f"\n{'='*60}")
    print(f"Progress: {current_index}/{len(patients)} "
          f"(completed: {len(all_results)}, skipped: {len(skipped)})")
    if current_index < len(patients):
        print(f"Next: {patients[current_index]['name']}")
        print(f"--- Cell 4 を実行して次の患者を読み込んでください ---")
    else:
        print(f"\n全患者の処理が完了しました！")
        print(f"--- Cell 6 を実行して集約 CSV を出力してください ---")
    print(f"{'='*60}")

---
## 6. 集約結果の出力

全症例の処理完了後に実行してください。`batch_summary.csv` に全結果が集約されます。

In [ ]:
# === 集約 CSV 出力 ===

print(f"{'='*60}")
print(f"BATCH SUMMARY")
print(f"{'='*60}")
print(f"  Completed: {len(all_results)}/{len(patients)}")
if skipped:
    print(f"  Skipped/Failed: {len(skipped)} ({', '.join(skipped)})")

if all_results:
    # CSV 出力
    summary_path = os.path.join(OUTPUT_DIR, "batch_summary.csv")
    fieldnames = list(dict.fromkeys(
        key for r in all_results for key in r.keys()
    ))

    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_results)

    print(f"\n  Summary CSV: {summary_path}")

    # 結果一覧
    print(f"\n  {'Patient':15s} {'Involvement':>12s} {'Healthy':>10s} {'GGO':>8s} {'Consol.':>8s} {'Dense':>8s}")
    print(f"  {'-'*65}")
    for r in all_results:
        inv = r.get('lung_involvement_pct', 0)
        # 各組織の割合を取得（キー名が異なる場合に対応）
        healthy = ggo = consol = dense = 0
        for k, v in r.items():
            kl = k.lower()
            if k.endswith('_pct'):
                if 'healthy' in kl:
                    healthy = v
                elif 'glass' in kl or 'ggo' in kl:
                    ggo = v
                elif 'consolidat' in kl:
                    consol = v
                elif 'dense' in kl:
                    dense = v
        print(f"  {r['patient']:15s} {inv:11.1f}% {healthy:9.1f}% {ggo:7.1f}% {consol:7.1f}% {dense:7.1f}%")

    print(f"\n{'='*60}")

    # Finder で開く
    import subprocess
    subprocess.call(['open', OUTPUT_DIR])
else:
    print("\n  No results to export.")

---
## 補足: 特定の患者のみ再処理

特定の患者を指定して再処理したい場合は、以下のセルで `current_index` を変更してから Cell 4 → 5 を実行してください。

In [ ]:
# 特定の患者に移動（番号は Cell 3 のリストで確認）
# 例: 5番目の患者に移動する場合
# current_index = 4  # 0始まりなので -1

# 患者名で検索する場合
# target = "03309523-1"
# current_index = next(i for i, p in enumerate(patients) if p["name"] == target)

print(f"Current index: {current_index}")
if current_index < len(patients):
    print(f"Next patient: {patients[current_index]['name']}")
print(f"\nTo change: uncomment and edit the lines above, then run Cell 4")